# EDA de imagens CT em Hounsfield Units

**Projeto Integrador — versão objetiva e reproduzível**

Este notebook realiza uma análise exploratória inicial de uma série de tomografia computadorizada do dataset **LIDC-IDRI**, disponibilizado pelo **The Cancer Imaging Archive (TCIA)**. O foco é verificar a distribuição dos valores em Hounsfield Units (HU), identificar o padding do campo de visão e criar um threshold simples como baseline para a etapa futura de segmentação pulmonar.

A análise não substitui métodos clínicos ou segmentação morfológica; ela documenta uma primeira inspeção dos dados para orientar o pipeline do projeto.


## 1. Objetivo e escopo

A análise responde a três perguntas:

1. A série baixada é uma série de CT e pode ser carregada como volume DICOM?
2. Como os valores HU estão distribuídos e quanto do volume corresponde ao padding?
3. Um intervalo simples de HU consegue indicar uma região pulmonar como baseline?

Para manter a execução leve e adequada ao Projeto Integrador, o notebook verifica somente uma série CT selecionada entre as primeiras séries do manifest.


In [ ]:
# No Colab, execute esta célula uma vez.
%pip install -q SimpleITK pydicom requests matplotlib numpy


In [ ]:
from pathlib import Path
import io
import os
import random
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import requests
import SimpleITK as sitk

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Ajuste estes caminhos conforme a organização do repositório.
DATA_DIR = Path('data')
MANIFEST_PATH = DATA_DIR / 'TCIA_LIDC-IDRI_20200921.tcia'
SERIES_DIR = DATA_DIR / 'serie_ct'
MAX_SERIES_TO_CHECK = 20

TCIA_METADATA_URL = 'https://services.cancerimagingarchive.net/nbia-api/services/v1/getSeriesMetaData'
TCIA_IMAGE_URL = 'https://services.cancerimagingarchive.net/nbia-api/services/v1/getImage'


## 2. Preparação e seleção da série

O manifest pode conter séries de modalidades diferentes. Por isso, a seleção consulta os metadados da TCIA e mantém apenas entradas cuja modalidade seja `CT`, antes de baixar os arquivos DICOM.

Coloque o arquivo `TCIA-LIDC-IDRI_20200921.tcia` na pasta `data/`. No Colab, também é possível fazer upload manual do manifest e ajustar `MANIFEST_PATH`.


In [ ]:
def read_series_uids(manifest_path: Path) -> list[str]:
    if not manifest_path.exists():
        raise FileNotFoundError(
            f'Manifest não encontrado em {manifest_path}. ' 
            'Adicione o arquivo à pasta data/ antes de executar.'
        )
    return [
        line.strip()
        for line in manifest_path.read_text(errors='ignore').splitlines()
        if line.strip().startswith('1.3.6.1.4.1')
    ]


def filter_ct_series(uids: list[str], limit: int = 20) -> list[str]:
    ct_uids = []
    for uid in uids[:limit]:
        response = requests.get(
            TCIA_METADATA_URL,
            params={'SeriesInstanceUID': uid},
            timeout=30,
        )
        response.raise_for_status()
        metadata = response.json()
        if metadata and metadata[0].get('Modality') == 'CT':
            ct_uids.append(uid)
        time.sleep(0.2)
    return ct_uids


uids = read_series_uids(MANIFEST_PATH)
series_ct = filter_ct_series(uids, MAX_SERIES_TO_CHECK)

print(f'Séries no manifest: {len(uids)}')
print(f'Séries CT verificadas: {len(series_ct)}')
if not series_ct:
    raise RuntimeError('Nenhuma série CT foi encontrada no limite analisado.')


In [ ]:
def download_series(uid: str, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    if any(output_dir.iterdir()):
        print(f'Usando série já baixada em: {output_dir}')
        return output_dir

    response = requests.get(
        TCIA_IMAGE_URL,
        params={'SeriesInstanceUID': uid},
        timeout=180,
    )
    response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        archive.extractall(output_dir)
    print(f'Arquivos DICOM baixados: {len(list(output_dir.iterdir()))}')
    return output_dir


download_series(series_ct[0], SERIES_DIR)


## 3. Carregamento e conversão para HU

O `ImageSeriesReader` organiza os arquivos DICOM da série e aplica os metadados de escala. Em CT, a relação conceitual é `HU = pixel × RescaleSlope + RescaleIntercept`.


In [ ]:
reader = sitk.ImageSeriesReader()
dicom_files = reader.GetGDCMSeriesFileNames(str(SERIES_DIR))
if not dicom_files:
    raise RuntimeError('Nenhum arquivo DICOM foi encontrado na pasta da série.')
reader.SetFileNames(dicom_files)
volume = reader.Execute()
volume_array = sitk.GetArrayFromImage(volume)

print('Formato (fatias, altura, largura):', volume_array.shape)
print('HU mínimo:', int(volume_array.min()))
print('HU máximo:', int(volume_array.max()))
print('Espaçamento:', volume.GetSpacing())


## 4. Distribuição dos valores e visualização

O valor `-2048 HU` aparece como padding fora do campo de visão circular. Ele não representa tecido e, portanto, é excluído do histograma principal.


In [ ]:
PADDING_HU = -2048
padding_count = int(np.sum(volume_array == PADDING_HU))
total_voxels = volume_array.size
valid_mask = volume_array > -1500

print(f'Padding: {padding_count:,} voxels ({100 * padding_count / total_voxels:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(volume_array[valid_mask].ravel(), bins=120, range=(-1000, 1000), color='steelblue')
axes[0].axvline(-1000, color='gray', linestyle='--', label='Ar')
axes[0].axvline(0, color='navy', linestyle='--', label='Água')
axes[0].axvline(50, color='crimson', linestyle='--', label='Tecido/Osso')
axes[0].set(title='Distribuição de HU sem padding', xlabel='Hounsfield Units (HU)', ylabel='Frequência')
axes[0].legend()

central_slice = volume_array[volume_array.shape[0] // 2]
im = axes[1].imshow(central_slice, cmap='gray')
axes[1].set_title('Fatia central')
axes[1].axis('off')
fig.colorbar(im, ax=axes[1], label='HU', fraction=0.046)
plt.tight_layout()
plt.show()


## 5. Baseline de segmentação pulmonar

Como ponto de partida, usamos `-1000 < HU < -500`. Esse intervalo pode destacar regiões pulmonares, mas também pode incluir ar externo ou outras cavidades. Portanto, o resultado é apenas um baseline e deverá ser refinado com operações morfológicas e componentes conectados em etapas futuras.


In [ ]:
LOW_HU, HIGH_HU = -1000, -500
lung_baseline = (volume_array > LOW_HU) & (volume_array < HIGH_HU)
baseline_count = int(lung_baseline.sum())
print(f'Pixels na faixa pulmonar: {baseline_count:,} ({100 * baseline_count / total_voxels:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(central_slice, cmap='gray')
axes[0].set_title('Fatia central (HU)')
axes[0].axis('off')
axes[1].imshow(central_slice, cmap='gray')
axes[1].imshow(lung_baseline[volume_array.shape[0] // 2], cmap='Greens', alpha=0.4)
axes[1].set_title('Baseline: -1000 a -500 HU')
axes[1].axis('off')
plt.tight_layout()
plt.show()


## 6. Síntese dos achados

A série analisada possui estrutura tridimensional de CT e valores HU compatíveis com a escala esperada. O histograma concentra valores próximos ao ar pulmonar e aos tecidos de maior densidade. O padding representa uma parcela relevante do volume e deve ser removido ou mascarado antes de análises posteriores.

O intervalo de `-1000` a `-500 HU` é útil como referência inicial, mas não isola sozinho os pulmões. A próxima etapa recomendada é combinar o threshold com restrição ao corpo do paciente, operações morfológicas e análise de componentes conectados.

> **Conclusão:** o EDA fornece uma validação inicial dos dados e define um baseline simples para orientar a segmentação pulmonar, sem apresentar o threshold como resultado final.


## Referências

[1] [The Cancer Imaging Archive — LIDC-IDRI](https://www.cancerimagingarchive.net/collection/lidc-idri/)

[2] [TCIA NBIA REST API](https://nbia.cancerimagingarchive.net/nbia-search/)

[3] [SimpleITK — Image Series Reader](https://simpleitk.readthedocs.io/en/master/link_DicomSeriesReader_docs.html)
